# EternityQuant — Kaggle ML 训练

在 Kaggle 的免费 GPU（T4/P100）上训练 EternityQuant 的量化选股模型。

## 支持算法
- **LightGBM** (CPU) — 基线模型，qlib Alpha158 特征
- **MLP** (CUDA) — 自写全连接网络，158→512→256→128→1
- **GRU** (CUDA) — 自写时序模型，6×26 时序重塑，**量化选股最佳**
- **LSTM** (CUDA) — 与 GRU 同架构，3 门替代 2 门
- **DeepLOB** (CUDA) — CNN+BiLSTM+Attention，顶会论文复现
- **TFT** (CUDA) — Temporal Fusion Transformer，Google 多时间跨度预测

## v0.20 关键升级
- **统一数据目录**：所有数据集中到 `~/.eternityquant/data/{a,hk,us}/` 下
- **权威标准化处理器**：对标 qlib 官方 benchmarks，特征走 `RobustZScoreNorm(clip_outlier=True)`，标签走 `CSZScoreNorm`/`CSRankNorm`（MAD 抗异常值）

## 多卡 GPU 支持
Kaggle 单台只有 1 张 GPU，但代码支持 `gpu_ids="0,1,2,3"` 参数，
在租用多 GPU 云服务器时自动启用 `nn.DataParallel` 并行训练。

## 输出
- 训练好的模型文件（`.pkl`）→ Kaggle Output 中下载
- 训练指标（IC、ICIR、Rank IC）

---

## 1. 环境准备

Kaggle 预装 PyTorch CUDA，只需安装 qlib 和 lightgbm。

In [ ]:
# @title 安装依赖
import sys, subprocess

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

# qlib — PyPI 版 0.9.7 有两个 bug，从 GitHub master 安装
# (issue #1949: inst_processors 双重复值 + Corr 算子 series_right 为空)
# 注意：git+https 安装没有 .git 目录，需 SETUPTOOLS_SCM_PRETEND_VERSION 跳过版本检测
import os
_env = os.environ.copy()
_env["SETUPTOOLS_SCM_PRETEND_VERSION"] = "1.0.0"
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/microsoft/qlib.git", "--no-build-isolation"],
        env=_env, stderr=subprocess.DEVNULL)
    print("  ✓ qlib GitHub 源码安装成功")
except subprocess.CalledProcessError:
    _pip("joblib==1.2.0", "pyqlib>=0.9")
    print("  ⚠ qlib 源码安装失败，使用 PyPI 版 + joblib 降级绕开")

# LightGBM
_pip("lightgbm>=4.0")

# akshare（成分股列表获取）
_pip("akshare>=1.10")

# PyTorch (Kaggle 预装 CUDA 版，只需确认)
import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("依赖安装完成 ✓")

In [ ]:
# @title 克隆 EternityQuant
import os
REPO_URL = "https://github.com/Eternity231/EternityQuant.git"
PROJECT_DIR = "/kaggle/working/EternityQuant"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

# 关键：设置 EQ_HOME 指向用户主目录 ~/.eternityquant
# 否则数据和模型会落到项目内 .eternityquant/，与 tar 包解压路径不一致
os.environ["EQ_HOME"] = os.path.expanduser("~/.eternityquant")
os.makedirs(os.environ["EQ_HOME"], exist_ok=True)
print(f"EQ_HOME = {os.environ['EQ_HOME']}")

os.chdir(PROJECT_DIR)
!pip install -e . --quiet
print(f"工作目录: {os.getcwd()}")

## 2. 准备 qlib 数据

### 统一数据目录
所有数据集中到项目内 `.eternityquant/data/a/qlib_cn_data/` 下。
在 Kaggle 上路径为 `/kaggle/working/EternityQuant/.eternityquant/data/a/qlib_cn_data/`。

### 方案 A：上传本地打包的 `cn_data.tar.gz`（推荐）
将本地打包好的 `cn_data.tar.gz` 上传为 Kaggle Dataset，
然后在本 Notebook 右侧 Add Data 中添加该 Dataset。

### 方案 B：在线拉取（慢）
Kaggle 网络可以访问腾讯 API，直接运行 `update_qlib_data` 拉取。

In [ ]:
# @title 方案 A：从 Kaggle Dataset 解压
import os, shutil
from pathlib import Path

# 统一目录：项目内 .eternityquant/data/a/qlib_cn_data/
PROJECT_DIR = "/kaggle/working/EternityQuant"
QLIB_TARGET = Path(PROJECT_DIR) / ".eternityquant" / "data" / "a" / "qlib_cn_data"
QLIB_TARGET.mkdir(parents=True, exist_ok=True)

# Kaggle Dataset 挂载路径
KAGGLE_DATASET = "/kaggle/input/qlib-cn-data"
TAR_NAME = "cn_data.tar.gz"

src_tar = os.path.join(KAGGLE_DATASET, TAR_NAME)
if os.path.exists(src_tar):
    !tar xzf "{src_tar}" -C {QLIB_TARGET}/
    print(f"已从 Kaggle Dataset 解压到 {QLIB_TARGET}")
elif os.path.exists(TAR_NAME):
    !tar xzf {TAR_NAME} -C {QLIB_TARGET}/
    print(f"已从当前目录解压到 {QLIB_TARGET}")
else:
    print(f"未找到 {TAR_NAME}，请添加 Kaggle Dataset 或切换到方案 B")

# 验证
cal = QLIB_TARGET / "calendars" / "day.txt"
if cal.exists():
    print(f"✓ 数据就绪: {cal}")
    n_features = len(list((QLIB_TARGET / "features").iterdir())) if (QLIB_TARGET / "features").exists() else 0
    print(f"  特征目录: {n_features} 只股票")
else:
    print("✗ 数据未找到")

## 3. 港股数据（可选）

如果要在 Kaggle 训练港股模型，把本地打包的 `hk_data.tar.gz` 上传为 Kaggle Dataset，
然后在右侧 Add Data 中添加该 Dataset。

本地打包命令：
```bash
tar czf hk_data.tar.gz -C ~/.eternityquant/data/hk .
```

In [ ]:
# @title 解压 hk_data.tar.gz 到统一港股目录
import os
from pathlib import Path

# 统一目录：项目内 .eternityquant/data/hk/
PROJECT_DIR = "/kaggle/working/EternityQuant"
HK_TARGET = Path(PROJECT_DIR) / ".eternityquant" / "data" / "hk"
HK_TARGET.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "/kaggle/input/qlib-cn-data"
src_tar = os.path.join(KAGGLE_DATASET, "hk_data.tar.gz")
if os.path.exists(src_tar):
    !tar xzf "{src_tar}" -C {HK_TARGET}/
    print(f"✓ 已从 Kaggle Dataset 解压到 {HK_TARGET}")
elif os.path.exists("hk_data.tar.gz"):
    !tar xzf hk_data.tar.gz -C {HK_TARGET}/
    print(f"✓ 已从当前目录解压到 {HK_TARGET}")
else:
    print("未找到 hk_data.tar.gz，跳过港股数据（仅训练 A 股可忽略此单元格）")

# 验证港股数据
for sub in ("daily", "5m", "1m"):
    d = HK_TARGET / sub
    if d.exists():
        n = len(list(d.iterdir()))
        print(f"  hk/{sub}: {n} 个文件")
    else:
        print(f"  hk/{sub}: 未找到")

In [ ]:
# @title 方案 B：在线拉取 A 股数据（腾讯 API，自动落到项目内 .eternityquant）
import os
from pathlib import Path

PROJECT_DIR = "/kaggle/working/EternityQuant"
QLIB_DATA_DIR = Path(PROJECT_DIR) / ".eternityquant" / "data" / "a" / "qlib_cn_data"

if not QLIB_DATA_DIR.exists() or not (QLIB_DATA_DIR / "calendars").exists():
    os.chdir("/kaggle/working/EternityQuant")
    from eq.strategy.factors.ml_data_updater import update_qlib_data
    result = update_qlib_data(
        start="2015-01-01",  # 拉近 10 年数据
        universe="csi300",
        workers=5,
        verbose=True,
    )
    print(f"数据更新完成: {result}")
else:
    print(f"qlib 数据已存在: {QLIB_DATA_DIR}")

# 验证
cal = QLIB_DATA_DIR / "calendars" / "day.txt"
if cal.exists():
    with open(cal) as f:
        lines = f.read().strip().split("\n")
    print(f"✓ 日历: {len(lines)} 个交易日  {lines[0]} ~ {lines[-1]}")
feat_dir = QLIB_DATA_DIR / "features"
if feat_dir.exists():
    n = len(list(feat_dir.iterdir()))
    print(f"✓ 特征: {n} 只股票")
else:
    print("✗ 特征目录未找到")

In [ ]:
# @title 迁移旧目录数据到统一目录（如有旧数据）
import os
os.chdir("/kaggle/working/EternityQuant")
from eq.data.paths import migrate_legacy_data_layout
result = migrate_legacy_data_layout(verbose=True)
print(f"\n迁移完成: 复制 {len(result['copied'])} 项，跳过 {len(result['skipped'])} 项")
print("统一目录结构：")
print("  ~/.eternityquant/data/a/qlib_cn_data/  (A 股 qlib)")
print("  ~/.eternityquant/data/hk/              (港股)")
print("  ~/.eternityquant/data/us/              (美股)")

## 3. 训练模型

### v0.20 权威标准化处理器链
对标 qlib 官方 benchmarks（GRU/ALSTM/LightGBM）同款配置：

| 路径 | 特征处理器 (infer) | 标签处理器 (learn) |
|------|-------------------|-------------------|
| `train()` (LightGBM) | ProcessInf → **RobustZScoreNorm(clip)** → Fillna | DropnaLabel → **CSZScoreNorm** |
| `train_torch()` (RNN/DeepLOB/TFT) | 同上 | DropnaLabel → **CSRankNorm** |

**关键升级**：`RobustZScoreNorm` 用 MAD（中位绝对偏差）替代 std，抗异常值；`CSRankNorm` 横截面排序归一化，免疫异常值。处理器参数在 `fit_start_time`/`fit_end_end` 范围内学习，绝无数据泄露。

### 3.1 LightGBM (CPU 基线)

In [ ]:
# @title 训练 LightGBM
import sys, os
os.chdir("/kaggle/working/EternityQuant")
sys.path.insert(0, ".")

from eq.strategy.factors.ml_workflow import train

result = train(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="lightgbm",
    device="cpu",
    name="kaggle_lgbm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.2 MLP (CUDA)

In [ ]:
# @title 训练 MLP
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="mlp",
    device="cuda",
    name="kaggle_mlp_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.3 GRU (CUDA) — 推荐，量化选股最佳

In [ ]:
# @title 训练 GRU (GPU 优化参数)
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# T4/P100 16GB 优化参数: batch_size=1024, hidden_size=128, num_layers=2
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="gru",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=1024,
    name="kaggle_gru_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  Epochs: {result['metrics']['epochs']}")
print(f"  模型文件: {result['model_path']}")

### 3.4 LSTM (CUDA)

In [ ]:
# @title 训练 LSTM (GPU 优化参数)
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="lstm",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=1024,
    name="kaggle_lstm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.5 DeepLOB (CUDA) — CNN+BiLSTM+Attention

[Zhang et al., 2019] 针对金融微观结构设计的专用架构。
卷积层提取空间特征，BiLSTM 建模时序，注意力机制加权。

In [ ]:
# @title 训练 DeepLOB (GPU 优化参数)
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# GPU 16GB 优化参数: batch_size=256, seq_len=120, hidden=64, dropout=0.3
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="deeplob",
    device="cuda",
    optimizer="adamw",
    loss_type="sharpe",
    batch_size=256,
    seq_len=120,
    dropout=0.3,
    name="kaggle_deeplob_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.6 TFT (CUDA) — Temporal Fusion Transformer

[Lim et al., 2019] Google 多时间跨度预测模型。
多头注意力+GRN+位置编码，目前中低频时序最先进模型之一。

In [ ]:
# @title 训练 TFT (GPU 优化参数)
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# GPU 16GB 优化参数: batch_size=128, hidden_size=128, num_heads=4, dropout=0.4
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="tft",
    device="cuda",
    optimizer="adamw",
    loss_type="sharpe",
    hidden_size=128,
    num_heads=4,
    batch_size=128,
    dropout=0.4,
    name="kaggle_tft_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.7 高级优化器对比

| 优化器 | 核心思想 | 金融优势 |
|--------|---------|---------|
| SAM | 寻找平坦极小值 | 市场环境漂移时仍保持低损失 |
| Lookahead | 双权重：快权探索，慢权稳定 | 降低局部噪音带偏概率 |
| Lion | 只看梯度符号，忽略幅度 | 免疫闪崩等极端异常值 |

In [ ]:
# @title 训练 GRU + SAM 优化器 + 可微夏普比率
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="gru",
    device="cuda",
    optimizer="sam",       # Sharpness-Aware Minimization
    loss_type="sharpe",    # 可微夏普比率损失
    name="kaggle_gru_sam_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

In [ ]:
# @title 训练 GRU + Lion 优化器
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="gru",
    device="cuda",
    optimizer="lion",     # EvoLved Sign Momentum
    loss_type="sharpe",
    name="kaggle_gru_lion_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.8 对抗训练 + 特征正交化

FGSM 对抗训练 + 特征正交化去 Beta，提升模型在实盘异常行情中的鲁棒性。

In [ ]:
# @title 训练 TFT + 对抗训练 + 特征正交化
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="tft",
    device="cuda",
    optimizer="adamw",
    loss_type="sharpe",
    adversarial=True,       # FGSM 对抗训练
    orthogonalize=True,     # 特征正交化去 Beta
    name="kaggle_tft_adv_orth_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.9 多卡并行训练（多 GPU 服务器专用）

Kaggle 只有 1 张 GPU，但如果租用多 GPU 云服务器，
可以用 `gpu_ids` 参数启用 `nn.DataParallel` 自动并行：

```python
# 双卡训练 TFT
train_torch(universe="csi300", horizon=5, algo="tft", device="cuda",
            gpu_ids="0,1", name="multigpu_tft")

# 四卡训练 DeepLOB
train_torch(universe="csi300", horizon=5, algo="deeplob", device="cuda",
            gpu_ids="0,1,2,3", name="multigpu_deeplob")
```

### 3.10 超参搜索（自动找最佳 LSTM/GRU 参数）

In [ ]:
# @title GRU 超参网格搜索
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import search_lstm

results = search_lstm(
    universe="csi300",
    horizon=5,
    device="cuda",
    fast=True,        # 快速模式：每组合 max_steps=50
    auto_train=True,  # 搜索完自动用最佳参数全量训练
    algo="gru",      # 可选 "lstm" 或 "gru"
)

print(f"\n搜索完成 {len(results)} 组合")
for i, r in enumerate(results[:3]):
    print(f"  #{i+1}: hidden={r['hidden_size']} layers={r['num_layers']} "
          f"lr={r['lr']} batch={r['batch_size']}  IC={r['ic']:+.4f}")

### 3.11 港股模型训练（GRU 多频率）

港股训练不走 qlib，用 `eq.data.hk_market` 的自写特征 + `_SimpleSeqModel`。

支持三个频率分别训练：
- `train_hk()` — 日线，horizon=5（5 日后收益）
- `train_hk_minute(freq="5m")` — 5 分钟线，horizon=30（2.5 小时后）
- `train_hk_minute(freq="1m")` — 1 分钟线，horizon=30（30 分钟后）

训练后可用 `predict_hk_ensemble()` 做多频率集成预测。

In [ ]:
# @title 训练港股日线 GRU
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.data.hk_market import train_hk

result = train_hk(
    top_n=100,
    horizon=5,
    hidden_size=128,
    num_layers=2,
    cell_type="gru",
    batch_size=512,
    max_steps=200,
    device="cuda",
    dropout=0.3,
    walk_forward=True,
    name="kaggle_hk_gru_h5",
)
print(f"\n港股日线训练完成:")
print(f"  IC: {result['ic']:+.4f}")
print(f"  模型: {result['model_path']}")
print(f"  样本: {result['train_samples']}")

In [ ]:
# @title 训练港股 5 分钟线 GRU
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.data.hk_market import train_hk_minute

result = train_hk_minute(
    top_n=50,
    freq="5m",
    horizon=30,
    hidden_size=128,
    num_layers=2,
    cell_type="gru",
    batch_size=512,
    max_steps=200,
    device="cuda",
    dropout=0.3,
    walk_forward=True,
    name="kaggle_hk_gru_5m_h30",
)
print(f"\n港股 5m 训练完成: IC={result['ic']:+.4f}  {result['model_path']}")

In [ ]:
# @title 训练港股 1 分钟线 GRU
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.data.hk_market import train_hk_minute

result = train_hk_minute(
    top_n=50,
    freq="1m",
    horizon=30,
    hidden_size=128,
    num_layers=2,
    cell_type="gru",
    batch_size=512,
    max_steps=200,
    device="cuda",
    dropout=0.3,
    walk_forward=True,
    name="kaggle_hk_gru_1m_h30",
)
print(f"\n港股 1m 训练完成: IC={result['ic']:+.4f}  {result['model_path']}")

In [ ]:
# @title 港股多频率集成预测 Top 10
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.data.hk_market import predict_hk_ensemble
from pathlib import Path

models_dir = Path.home() / ".eternityquant" / "data" / "hk" / "models"
daily_pkl = str(models_dir / "kaggle_hk_gru_h5.pkl")
five_m_pkl = str(models_dir / "kaggle_hk_gru_5m_h30.pkl")
one_m_pkl = str(models_dir / "kaggle_hk_gru_1m_h30.pkl")

top_df = predict_hk_ensemble(
    model_daily=daily_pkl,
    model_5m=five_m_pkl if os.path.exists(five_m_pkl) else None,
    model_1m=one_m_pkl if os.path.exists(one_m_pkl) else None,
    top_n=10,
    weights={"daily": 0.5, "5m": 0.3, "1m": 0.2},
)
print("\n港股多频率集成预测 Top 10:")
print(top_df.to_string(index=False))

## 4. 导出模型回本地

训练好的模型文件在 `~/.eternityquant/ml_models/` 目录下（Kaggle 上即 `/root/.eternityquant/ml_models/`）。
Kaggle Notebook 运行结束后，Output 中的文件可下载。

In [ ]:
# @title 列出所有训练好的模型
import os
from pathlib import Path
models_dir = Path.home() / ".eternityquant" / "ml_models"
if models_dir.exists():
    for f in sorted(models_dir.iterdir()):
        size = f.stat().st_size
        print(f"  {f.name:45s}  {size/1024:.1f} KB")
else:
    print("未找到模型文件")

In [ ]:
# @title 复制模型到 Kaggle Output（运行结束后可下载）
import os, shutil
from pathlib import Path
models_dir = Path.home() / ".eternityquant" / "ml_models"
output_dir = Path("/kaggle/working/models_output")
output_dir.mkdir(parents=True, exist_ok=True)

if models_dir.exists() and any(models_dir.iterdir()):
    for f in models_dir.iterdir():
        shutil.copy2(f, output_dir / f.name)
    print(f"✓ 已复制 {len(list(output_dir.iterdir()))} 个模型到 {output_dir}")
    print("Kaggle Notebook 运行结束后，在 Output 标签页下载这些文件")
else:
    print("没有模型文件可导出")

## 5. 回本地导入模型

```bash
# 解压模型文件（从 Kaggle Output 下载的 models_output.zip）
unzip models_output.zip -d ~/.eternityquant/ml_models/

# 登记模型
eq ml register --name "kaggle_gru_csi300_h5" --universe csi300 \
    --features "Alpha158" --algo gru --horizon 5 \
    --train-period "2015-01-01~2024-12-31" \
    --model-path ~/.eternityquant/ml_models/torch_gru_csi300_5d.pkl \
    --metrics '{"ic": 0.15}' \
    --notes "Kaggle T4/P100 GPU 训练"

# 激活模型
eq ml activate <model_id>

# 批量预测今日 Top 10
eq ml predict-batch <model_id> --top 10
```

---

## 附：Kaggle GPU 配置信息

In [ ]:
# @title GPU 信息
import torch, psutil, platform
print(f"系统: {platform.platform()}")
print(f"CPU: {psutil.cpu_count()} 核")
print(f"内存: {psutil.virtual_memory().total / 1e9:.1f} GB")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU: 不可用（将使用 CPU）")